In [3]:
import pandas as pd
import matplotlib.pyplot as plt


In [4]:
df = pd.read_csv("../data/train.csv")

df.shape

(159571, 8)

In [5]:
X = df["comment_text"]

label_columns = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

y = df[label_columns]

In [6]:
X.head()

0    Explanation\nWhy the edits made under my usern...
1    D'aww! He matches this background colour I'm s...
2    Hey man, I'm really not trying to edit war. It...
3    "\nMore\nI can't make any real suggestions on ...
4    You, sir, are my hero. Any chance you remember...
Name: comment_text, dtype: str

In [7]:
y.head()

,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0,0,0,0,0,0
1,0,0,0,0,0,0
2,0,0,0,0,0,0
3,0,0,0,0,0,0
4,0,0,0,0,0,0


In [8]:
from sklearn.model_selection import train_test_split

In [9]:
is_harmful = (y.sum(axis=1) > 0).astype(int)

is_harmful.value_counts()

0    143346
1     16225
Name: count, dtype: int64

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=is_harmful
)

In [11]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (127656,)
X_test: (31915,)
y_train: (127656, 6)
y_test: (31915, 6)


In [12]:
train_harmful_percentage = (y_train.sum(axis=1) > 0).mean() * 100
test_harmful_percentage = (y_test.sum(axis=1) > 0).mean() * 100

print("Training set harmful percentage:", train_harmful_percentage)
print("Testing set harmful percentage:", test_harmful_percentage)

Training set harmful percentage: 10.167951369305008
Testing set harmful percentage: 10.167632774557418


In [13]:
label_distribution = pd.DataFrame({
    "Training %": y_train.mean() * 100,
    "Testing %": y_test.mean() * 100
}).round(2)

label_distribution

,Training %,Testing %
toxic,9.59,9.54
severe_toxic,1.01,0.96
obscene,5.32,5.21
threat,0.30,0.31
insult,4.94,4.94
identity_hate,0.90,0.81


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [15]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=20000
)

In [16]:
X_train_tfidf = tfidf.fit_transform(X_train)

In [17]:
X_test_tfidf = tfidf.transform(X_test)

In [18]:
print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (127656, 20000)
Testing TF-IDF shape: (31915, 20000)


In [19]:
feature_names = tfidf.get_feature_names_out()

feature_names[:20]

array(['00', '000', '000 000', '000000', '01', '02', '03', '04', '05',
       '06', '07', '08', '084080', '084080 background', '09', '0px', '10',
       '10 000', '10 years', '100'], dtype=object)

In [20]:
len(feature_names)

20000

In [21]:
print(X_train.iloc[0])

Message
Thanks for the message. Have fun out there!  22:01, 18 Feb 2005 (UTC)


In [22]:
X_train_tfidf[0]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 21 stored elements and shape (1, 20000)>

In [23]:
first_comment_vector = X_train_tfidf[0]

nonzero_indices = first_comment_vector.nonzero()[1]

first_comment_features = pd.DataFrame({
    "Feature": feature_names[nonzero_indices],
    "TF-IDF Score": first_comment_vector[0, nonzero_indices].toarray().flatten()
})

first_comment_features.sort_values(
    "TF-IDF Score",
    ascending=False
)

,Feature,TF-IDF Score
0,message,0.380233
19,feb 2005,0.310349
17,have fun,0.289017
11,feb,0.275927
16,the message,0.275502
18,out there,0.261790
9,01,0.249553
5,fun,0.240279
8,22,0.225653
20,2005 utc,0.225576


In [24]:
from sklearn.linear_model import LogisticRegression

In [25]:
toxic_model = LogisticRegression(
    max_iter=1000
)

In [26]:
toxic_model.fit(
    X_train_tfidf,
    y_train["toxic"]
)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

In [27]:
toxic_predictions = toxic_model.predict(X_test_tfidf)

toxic_predictions[:20]

array([0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0])

In [28]:
pd.Series(toxic_predictions).value_counts()

0    29941
1     1974
Name: count, dtype: int64

In [29]:
y_test["toxic"]

63958     0
11723     0
18282     1
145258    0
152834    0
         ..
94520     0
152761    0
44190     0
90128     0
38345     0
Name: toxic, Length: 31915, dtype: int64

In [30]:
y_test["toxic"].value_counts()

toxic
0    28869
1     3046
Name: count, dtype: int64

In [31]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test["toxic"],
    toxic_predictions
)

cm

array([[28706,   163],
       [ 1235,  1811]])

In [32]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_test["toxic"], toxic_predictions)
recall = recall_score(y_test["toxic"], toxic_predictions)
f1 = f1_score(y_test["toxic"], toxic_predictions)

print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1 Score:", round(f1, 3))

Precision: 0.917
Recall: 0.595
F1 Score: 0.722


In [33]:
toxic_probabilities = toxic_model.predict_proba(X_test_tfidf)[:, 1]

toxic_probabilities[:10]

array([0.01214147, 0.06403808, 0.93048439, 0.0175101 , 0.01015741,
       0.02879626, 0.02057326, 0.00415717, 0.00993251, 0.01113749])

In [34]:
models = {}

In [35]:
for label in label_columns:
    model = LogisticRegression(max_iter=1000)

    model.fit(
        X_train_tfidf,
        y_train[label]
    )

    models[label] = model

    print(f"Finished training: {label}")

Finished training: toxic
Finished training: severe_toxic
Finished training: obscene
Finished training: threat
Finished training: insult
Finished training: identity_hate


In [36]:
results = []

for label in label_columns:
    predictions = models[label].predict(X_test_tfidf)

    precision = precision_score(
        y_test[label],
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test[label],
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test[label],
        predictions,
        zero_division=0
    )

    results.append({
        "Label": label,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

In [37]:
results_df = pd.DataFrame(results)

results_df.round(3)

,Label,Precision,Recall,F1 Score
0,toxic,0.917,0.595,0.722
1,severe_toxic,0.530,0.233,0.323
2,obscene,0.912,0.608,0.730
3,threat,0.733,0.112,0.195
4,insult,0.834,0.521,0.641
5,identity_hate,0.684,0.201,0.310


In [38]:
results_df["Test Positives"] = [
    y_test[label].sum()
    for label in label_columns
]

results_df.round(3)

,Label,Precision,Recall,F1 Score,Test Positives
0,toxic,0.917,0.595,0.722,3046
1,severe_toxic,0.530,0.233,0.323,305
2,obscene,0.912,0.608,0.730,1662
3,threat,0.733,0.112,0.195,98
4,insult,0.834,0.521,0.641,1576
5,identity_hate,0.684,0.201,0.310,259


In [39]:
print("Macro Precision:", round(results_df["Precision"].mean(), 3))
print("Macro Recall:", round(results_df["Recall"].mean(), 3))
print("Macro F1:", round(results_df["F1 Score"].mean(), 3))

Macro Precision: 0.768
Macro Recall: 0.378
Macro F1: 0.487


## Baseline Model Results

The baseline model used TF-IDF text features and separate Logistic Regression classifiers for each harmful-content category.

The model achieved a macro precision of 0.768, macro recall of 0.378, and macro F1 score of 0.487.

Performance varied substantially across categories. The strongest F1 scores were observed for obscene (0.730), toxic (0.722), and insult (0.641). Performance was substantially weaker for severe toxic (0.323), identity hate (0.310), and threat (0.195).

The lower-performing categories also had considerably fewer positive examples in the test set. For example, the threat category contained only 98 positive test examples compared with 3,046 toxic examples. This suggests that class imbalance may contribute to the model's difficulty identifying rare harmful-content categories.

Overall, the baseline model showed relatively high precision but low recall, indicating that it was conservative when identifying harmful content and failed to detect many positive examples, particularly in the rarer categories.

In [102]:
train_is_harmful = (y_train.sum(axis=1) > 0).astype(int)

In [103]:
X_model_train, X_validation, y_model_train, y_validation = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=train_is_harmful
)

In [104]:
print("Model training:", X_model_train.shape)
print("Validation:", X_validation.shape)
print("Final test:", X_test.shape)

Model training: (102124,)
Validation: (25532,)
Final test: (31915,)


In [105]:
development_tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=20000
)

In [106]:
X_model_train_tfidf = development_tfidf.fit_transform(
    X_model_train
)

In [107]:
X_validation_tfidf = development_tfidf.transform(
    X_validation
)

print("Model training TF-IDF:", X_model_train_tfidf.shape)
print("Validation TF-IDF:", X_validation_tfidf.shape)

Model training TF-IDF: (102124, 20000)
Validation TF-IDF: (25532, 20000)


In [108]:
development_models = {}

for label in label_columns:
    model = LogisticRegression(max_iter=1000)

    model.fit(
        X_model_train_tfidf,
        y_model_train[label]
    )

    development_models[label] = model

    print(f"Finished training: {label}")

Finished training: toxic
Finished training: severe_toxic
Finished training: obscene
Finished training: threat
Finished training: insult
Finished training: identity_hate


In [109]:
validation_results = []

for label in label_columns:
    predictions = development_models[label].predict(
        X_validation_tfidf
    )

    precision = precision_score(
        y_validation[label],
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_validation[label],
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_validation[label],
        predictions,
        zero_division=0
    )

    validation_results.append({
        "Label": label,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

validation_results_df = pd.DataFrame(validation_results)

validation_results_df.round(3)

,Label,Precision,Recall,F1 Score
0,toxic,0.921,0.585,0.715
1,severe_toxic,0.509,0.218,0.305
2,obscene,0.930,0.599,0.729
3,threat,0.923,0.143,0.247
4,insult,0.825,0.527,0.643
5,identity_hate,0.818,0.187,0.304


In [110]:
print(
    "Validation Macro F1:",
    round(validation_results_df["F1 Score"].mean(), 3)
)

Validation Macro F1: 0.491


In [111]:
threat_probabilities = development_models["threat"].predict_proba(
    X_validation_tfidf
)[:, 1]

In [112]:
for threshold in [0.50, 0.40, 0.30, 0.20]:
    threat_predictions = (
        threat_probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation["threat"],
        threat_predictions,
        zero_division=0
    )

    recall = recall_score(
        y_validation["threat"],
        threat_predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_validation["threat"],
        threat_predictions,
        zero_division=0
    )

    print(
        f"Threshold {threshold}: "
        f"Precision={precision:.3f}, "
        f"Recall={recall:.3f}, "
        f"F1={f1:.3f}"
    )

Threshold 0.5: Precision=0.923, Recall=0.143, F1=0.247
Threshold 0.4: Precision=0.812, Recall=0.155, F1=0.260
Threshold 0.3: Precision=0.625, Recall=0.179, F1=0.278
Threshold 0.2: Precision=0.567, Recall=0.202, F1=0.298


In [113]:
for threshold in [0.20, 0.15, 0.10, 0.05]:
    threat_predictions = (
        threat_probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation["threat"],
        threat_predictions,
        zero_division=0
    )

    recall = recall_score(
        y_validation["threat"],
        threat_predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_validation["threat"],
        threat_predictions,
        zero_division=0
    )

    print(
        f"Threshold {threshold}: "
        f"Precision={precision:.3f}, "
        f"Recall={recall:.3f}, "
        f"F1={f1:.3f}"
    )

Threshold 0.2: Precision=0.567, Recall=0.202, F1=0.298
Threshold 0.15: Precision=0.556, Recall=0.238, F1=0.333
Threshold 0.1: Precision=0.509, Recall=0.321, F1=0.394
Threshold 0.05: Precision=0.373, Recall=0.488, F1=0.423


In [114]:
for threshold in [0.05, 0.04, 0.03, 0.02, 0.01]:
    threat_predictions = (
        threat_probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation["threat"],
        threat_predictions,
        zero_division=0
    )

    recall = recall_score(
        y_validation["threat"],
        threat_predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_validation["threat"],
        threat_predictions,
        zero_division=0
    )

    print(
        f"Threshold {threshold}: "
        f"Precision={precision:.3f}, "
        f"Recall={recall:.3f}, "
        f"F1={f1:.3f}"
    )

Threshold 0.05: Precision=0.373, Recall=0.488, F1=0.423
Threshold 0.04: Precision=0.317, Recall=0.536, F1=0.398
Threshold 0.03: Precision=0.251, Recall=0.583, F1=0.351
Threshold 0.02: Precision=0.191, Recall=0.690, F1=0.299
Threshold 0.01: Precision=0.095, Recall=0.810, F1=0.170


In [115]:
for threshold in [0.05, 0.06, 0.07, 0.08, 0.09, 0.10]:
    threat_predictions = (
        threat_probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation["threat"],
        threat_predictions,
        zero_division=0
    )

    recall = recall_score(
        y_validation["threat"],
        threat_predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_validation["threat"],
        threat_predictions,
        zero_division=0
    )

    print(
        f"Threshold {threshold}: "
        f"Precision={precision:.3f}, "
        f"Recall={recall:.3f}, "
        f"F1={f1:.3f}"
    )

Threshold 0.05: Precision=0.373, Recall=0.488, F1=0.423
Threshold 0.06: Precision=0.414, Recall=0.429, F1=0.421
Threshold 0.07: Precision=0.425, Recall=0.405, F1=0.415
Threshold 0.08: Precision=0.448, Recall=0.357, F1=0.397
Threshold 0.09: Precision=0.467, Recall=0.333, F1=0.389
Threshold 0.1: Precision=0.509, Recall=0.321, F1=0.394


In [116]:
thresholds = [
    i / 100
    for i in range(1, 51)
]

In [117]:
thresholds[:10]

[0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]

In [118]:
threshold_results = []

for label in label_columns:
    probabilities = development_models[label].predict_proba(
        X_validation_tfidf
    )[:, 1]

    best_threshold = 0
    best_f1 = 0
    best_precision = 0
    best_recall = 0

    for threshold in thresholds:
        predictions = (probabilities >= threshold).astype(int)

        precision = precision_score(
            y_validation[label],
            predictions,
            zero_division=0
        )

        recall = recall_score(
            y_validation[label],
            predictions,
            zero_division=0
        )

        f1 = f1_score(
            y_validation[label],
            predictions,
            zero_division=0
        )

        if f1 > best_f1:
            best_threshold = threshold
            best_f1 = f1
            best_precision = precision
            best_recall = recall

    threshold_results.append({
        "Label": label,
        "Best Threshold": best_threshold,
        "Precision": best_precision,
        "Recall": best_recall,
        "F1 Score": best_f1
    })

In [119]:
threshold_results_df = pd.DataFrame(threshold_results)

threshold_results_df.round(3)

,Label,Best Threshold,Precision,Recall,F1 Score
0,toxic,0.27,0.796,0.736,0.765
1,severe_toxic,0.15,0.414,0.552,0.473
2,obscene,0.21,0.822,0.770,0.795
3,threat,0.05,0.373,0.488,0.423
4,insult,0.19,0.669,0.769,0.716
5,identity_hate,0.17,0.586,0.369,0.453


In [120]:
tuned_macro_precision = threshold_results_df["Precision"].mean()
tuned_macro_recall = threshold_results_df["Recall"].mean()
tuned_macro_f1 = threshold_results_df["F1 Score"].mean()

print("Tuned Macro Precision:", round(tuned_macro_precision, 3))
print("Tuned Macro Recall:", round(tuned_macro_recall, 3))
print("Tuned Macro F1:", round(tuned_macro_f1, 3))

Tuned Macro Precision: 0.61
Tuned Macro Recall: 0.614
Tuned Macro F1: 0.604


In [121]:
comparison_df = pd.DataFrame({
    "Model": [
        "Default Threshold (0.50)",
        "Tuned Thresholds"
    ],
    "Macro Precision": [
        validation_results_df["Precision"].mean(),
        tuned_macro_precision
    ],
    "Macro Recall": [
        validation_results_df["Recall"].mean(),
        tuned_macro_recall
    ],
    "Macro F1": [
        validation_results_df["F1 Score"].mean(),
        tuned_macro_f1
    ]
})

comparison_df.round(3)

,Model,Macro Precision,Macro Recall,Macro F1
0,Default Threshold (0.50),0.821,0.376,0.491
1,Tuned Thresholds,0.610,0.614,0.604


### Threshold Tuning Results

The default classification threshold of 0.50 produced high macro precision (0.821) but relatively low macro recall (0.376), indicating that the baseline models were conservative and missed many harmful comments.

Classification thresholds were tuned separately for each harmful-content category using the validation set. The tuned thresholds improved macro recall from 0.376 to 0.614 and macro F1 from 0.491 to 0.604, while macro precision decreased from 0.821 to 0.610.

This represents approximately a 23% relative improvement in macro F1 compared with using the default threshold.

The results demonstrate the tradeoff between precision and recall in content moderation. Lower thresholds identify more potentially harmful comments but also increase false positives. For a human-in-the-loop moderation system, lower thresholds may be appropriate for identifying comments for REVIEW, while a higher-confidence threshold may be more appropriate before automatically HIDEing content.

Thresholds were selected using validation data rather than the final test set to avoid tuning model decisions directly on test performance.

In [122]:
balanced_threat_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

balanced_threat_model.fit(
    X_model_train_tfidf,
    y_model_train["threat"]
)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to

In [123]:
balanced_threat_predictions = balanced_threat_model.predict(
    X_validation_tfidf
)

balanced_threat_precision = precision_score(
    y_validation["threat"],
    balanced_threat_predictions,
    zero_division=0
)

balanced_threat_recall = recall_score(
    y_validation["threat"],
    balanced_threat_predictions,
    zero_division=0
)

balanced_threat_f1 = f1_score(
    y_validation["threat"],
    balanced_threat_predictions,
    zero_division=0
)

print("Precision:", round(balanced_threat_precision, 3))
print("Recall:", round(balanced_threat_recall, 3))
print("F1:", round(balanced_threat_f1, 3))

Precision: 0.263
Recall: 0.726
F1: 0.386


In [124]:
balanced_threat_probabilities = balanced_threat_model.predict_proba(
    X_validation_tfidf
)[:, 1]

for threshold in [0.50, 0.60, 0.70, 0.80, 0.90]:
    predictions = (
        balanced_threat_probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation["threat"],
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_validation["threat"],
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_validation["threat"],
        predictions,
        zero_division=0
    )

    print(
        f"Threshold {threshold}: "
        f"Precision={precision:.3f}, "
        f"Recall={recall:.3f}, "
        f"F1={f1:.3f}"
    )

Threshold 0.5: Precision=0.263, Recall=0.726, F1=0.386
Threshold 0.6: Precision=0.328, Recall=0.679, F1=0.442
Threshold 0.7: Precision=0.385, Recall=0.619, F1=0.475
Threshold 0.8: Precision=0.474, Recall=0.548, F1=0.508
Threshold 0.9: Precision=0.543, Recall=0.452, F1=0.494


In [125]:
balanced_threat_probabilities = balanced_threat_model.predict_proba(
    X_validation_tfidf
)[:, 1]

for threshold in [0.75, 0.78, 0.80, 0.82, 0.85, 0.88, 0.90]:
    predictions = (
        balanced_threat_probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation["threat"],
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_validation["threat"],
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_validation["threat"],
        predictions,
        zero_division=0
    )

    print(
        f"Threshold {threshold}: "
        f"Precision={precision:.3f}, "
        f"Recall={recall:.3f}, "
        f"F1={f1:.3f}"
    )

Threshold 0.75: Precision=0.422, Recall=0.583, F1=0.490
Threshold 0.78: Precision=0.440, Recall=0.571, F1=0.497
Threshold 0.8: Precision=0.474, Recall=0.548, F1=0.508
Threshold 0.82: Precision=0.479, Recall=0.536, F1=0.506
Threshold 0.85: Precision=0.484, Recall=0.524, F1=0.503
Threshold 0.88: Precision=0.525, Recall=0.500, F1=0.512
Threshold 0.9: Precision=0.543, Recall=0.452, F1=0.494


In [126]:
balanced_models = {}

for label in label_columns:
    model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    )

    model.fit(
        X_model_train_tfidf,
        y_model_train[label]
    )

    balanced_models[label] = model

    print(f"Finished training: {label}")

Finished training: toxic
Finished training: severe_toxic
Finished training: obscene
Finished training: threat
Finished training: insult
Finished training: identity_hate


In [127]:
balanced_thresholds = [
    i / 100
    for i in range(1, 100)
]

In [128]:
balanced_threshold_results = []

for label in label_columns:
    probabilities = balanced_models[label].predict_proba(
        X_validation_tfidf
    )[:, 1]

    best_threshold = 0
    best_f1 = 0
    best_precision = 0
    best_recall = 0

    for threshold in balanced_thresholds:
        predictions = (probabilities >= threshold).astype(int)

        precision = precision_score(
            y_validation[label],
            predictions,
            zero_division=0
        )

        recall = recall_score(
            y_validation[label],
            predictions,
            zero_division=0
        )

        f1 = f1_score(
            y_validation[label],
            predictions,
            zero_division=0
        )

        if f1 > best_f1:
            best_threshold = threshold
            best_f1 = f1
            best_precision = precision
            best_recall = recall

    balanced_threshold_results.append({
        "Label": label,
        "Best Threshold": best_threshold,
        "Precision": best_precision,
        "Recall": best_recall,
        "F1 Score": best_f1
    })

In [129]:
balanced_threshold_results_df = pd.DataFrame(
    balanced_threshold_results
)

balanced_threshold_results_df.round(3)

,Label,Best Threshold,Precision,Recall,F1 Score
0,toxic,0.70,0.769,0.758,0.763
1,severe_toxic,0.92,0.385,0.556,0.455
2,obscene,0.78,0.810,0.776,0.792
3,threat,0.89,0.539,0.488,0.512
4,insult,0.80,0.710,0.738,0.723
5,identity_hate,0.83,0.401,0.556,0.466


In [130]:
balanced_macro_precision = balanced_threshold_results_df["Precision"].mean()
balanced_macro_recall = balanced_threshold_results_df["Recall"].mean()
balanced_macro_f1 = balanced_threshold_results_df["F1 Score"].mean()

print("Balanced Macro Precision:", round(balanced_macro_precision, 3))
print("Balanced Macro Recall:", round(balanced_macro_recall, 3))
print("Balanced Macro F1:", round(balanced_macro_f1, 3))

Balanced Macro Precision: 0.602
Balanced Macro Recall: 0.645
Balanced Macro F1: 0.619


In [131]:
hybrid_results = []

for label in label_columns:
    original_row = threshold_results_df[
        threshold_results_df["Label"] == label
    ].iloc[0]

    balanced_row = balanced_threshold_results_df[
        balanced_threshold_results_df["Label"] == label
    ].iloc[0]

    if balanced_row["F1 Score"] > original_row["F1 Score"]:
        selected_model = "Balanced"
        selected_row = balanced_row
    else:
        selected_model = "Original"
        selected_row = original_row

    hybrid_results.append({
        "Label": label,
        "Selected Model": selected_model,
        "Threshold": selected_row["Best Threshold"],
        "Precision": selected_row["Precision"],
        "Recall": selected_row["Recall"],
        "F1 Score": selected_row["F1 Score"]
    })

In [132]:
hybrid_results_df = pd.DataFrame(hybrid_results)

hybrid_results_df.round(3)

,Label,Selected Model,Threshold,Precision,Recall,F1 Score
0,toxic,Original,0.27,0.796,0.736,0.765
1,severe_toxic,Original,0.15,0.414,0.552,0.473
2,obscene,Original,0.21,0.822,0.770,0.795
3,threat,Balanced,0.89,0.539,0.488,0.512
4,insult,Balanced,0.80,0.710,0.738,0.723
5,identity_hate,Balanced,0.83,0.401,0.556,0.466


In [133]:
hybrid_macro_precision = hybrid_results_df["Precision"].mean()
hybrid_macro_recall = hybrid_results_df["Recall"].mean()
hybrid_macro_f1 = hybrid_results_df["F1 Score"].mean()

print("Hybrid Macro Precision:", round(hybrid_macro_precision, 3))
print("Hybrid Macro Recall:", round(hybrid_macro_recall, 3))
print("Hybrid Macro F1:", round(hybrid_macro_f1, 3))

Hybrid Macro Precision: 0.614
Hybrid Macro Recall: 0.64
Hybrid Macro F1: 0.623


In [134]:
model_comparison_df = pd.DataFrame({
    "Approach": [
        "Default Threshold (0.50)",
        "Tuned Original Models",
        "Tuned Balanced Models",
        "Hybrid Models"
    ],
    "Macro Precision": [
        validation_results_df["Precision"].mean(),
        tuned_macro_precision,
        balanced_macro_precision,
        hybrid_macro_precision
    ],
    "Macro Recall": [
        validation_results_df["Recall"].mean(),
        tuned_macro_recall,
        balanced_macro_recall,
        hybrid_macro_recall
    ],
    "Macro F1": [
        validation_results_df["F1 Score"].mean(),
        tuned_macro_f1,
        balanced_macro_f1,
        hybrid_macro_f1
    ]
})

model_comparison_df.round(3)

,Approach,Macro Precision,Macro Recall,Macro F1
0,Default Threshold (0.50),0.821,0.376,0.491
1,Tuned Original Models,0.610,0.614,0.604
2,Tuned Balanced Models,0.602,0.645,0.619
3,Hybrid Models,0.614,0.640,0.623


### Hybrid Model Results

Class-weight balancing did not improve every harmful-content category equally. The original Logistic Regression models achieved higher validation F1 scores for toxic, severe toxic, and obscene, while the balanced models performed better for threat, insult, and identity hate.

Based on validation performance, a hybrid approach was created by selecting the model version with the higher F1 score for each category. The hybrid approach achieved a macro precision of 0.614, macro recall of 0.640, and macro F1 score of 0.623.

Compared with the tuned original models, the hybrid approach improved all three macro metrics. Compared with the default 0.50 threshold baseline, macro F1 increased from 0.491 to 0.623, an improvement of approximately 27%.

These results suggest that class imbalance affects the harmful-content categories differently. Rather than applying the same modeling strategy to every category, selecting the model and classification threshold separately for each category produced better overall validation performance.

In [135]:
threat_probabilities = balanced_models["threat"].predict_proba(
    X_validation_tfidf
)[:, 1]

threat_predictions = (
    threat_probabilities >= 0.89
).astype(int)

threat_error_analysis = pd.DataFrame({
    "comment": X_validation.values,
    "actual": y_validation["threat"].values,
    "predicted": threat_predictions,
    "probability": threat_probabilities
})

In [136]:
threat_false_positives = threat_error_analysis[
    (threat_error_analysis["actual"] == 0) &
    (threat_error_analysis["predicted"] == 1)
].sort_values(
    "probability",
    ascending=False
)

threat_false_positives.head(10)

,comment,actual,predicted,probability
23100,Ahem \n\nDIE. NOW. kthxbye.,0,1,0.999928
5586,You are a raging faggot. Kill yourself.,0,1,0.999560
6158,Dahlia Tal does die \n\nDahlia Tal does die. I...,0,1,0.999491
24075,stupid \n\nassholes never die rightß INSTEAD; ...,0,1,0.999260
22608,die \n\nplease why dont u do us all a favioe a...,0,1,0.999218
8666,"you gay motherfucker i know where you live,i a...",0,1,0.998812
18233,"Where do you live, you little shit? \n\nGive u...",0,1,0.998700
574,Gay\nEveryone is e-mailing me about deep shado...,0,1,0.998342
1969,Fuck off and die you old timer,0,1,0.996225
11247,) \n\nFuck you. Eat shit and die.,0,1,0.994140


In [137]:
for label in label_columns:
    threat_error_analysis[label] = (
        y_validation[label]
        .reset_index(drop=True)
    )

In [138]:
threat_false_positives = threat_error_analysis[
    (threat_error_analysis["actual"] == 0) &
    (threat_error_analysis["predicted"] == 1)
].sort_values(
    "probability",
    ascending=False
)

threat_false_positives[
    [
        "comment",
        "probability",
        "toxic",
        "severe_toxic",
        "obscene",
        "threat",
        "insult",
        "identity_hate"
    ]
].head(10)

,comment,probability,toxic,severe_toxic,obscene,threat,insult,identity_hate
23100,Ahem \n\nDIE. NOW. kthxbye.,0.999928,1,0,0,0,0,0
5586,You are a raging faggot. Kill yourself.,0.999560,1,0,1,0,1,0
6158,Dahlia Tal does die \n\nDahlia Tal does die. I...,0.999491,1,0,0,0,0,0
24075,stupid \n\nassholes never die rightß INSTEAD; ...,0.999260,1,1,1,0,0,0
22608,die \n\nplease why dont u do us all a favioe a...,0.999218,1,1,0,0,0,0
8666,"you gay motherfucker i know where you live,i a...",0.998812,1,1,1,0,1,1
18233,"Where do you live, you little shit? \n\nGive u...",0.998700,1,1,1,0,0,0
574,Gay\nEveryone is e-mailing me about deep shado...,0.998342,1,1,1,0,1,1
1969,Fuck off and die you old timer,0.996225,1,1,1,0,1,0
11247,) \n\nFuck you. Eat shit and die.,0.994140,1,0,1,0,1,0


In [139]:
other_harmful_labels = [
    "toxic",
    "severe_toxic",
    "obscene",
    "insult",
    "identity_hate"
]

threat_false_positives["other_harmful"] = (
    threat_false_positives[other_harmful_labels].sum(axis=1) > 0
)

threat_false_positives["other_harmful"].value_counts()

other_harmful
True     34
False     1
Name: count, dtype: int64

In [140]:
threat_false_positives["other_harmful"].value_counts(
    normalize=True
).mul(100).round(2)

other_harmful
True     97.14
False     2.86
Name: proportion, dtype: float64

In [141]:
threat_error_analysis["actual_harmful"] = (
    threat_error_analysis[label_columns].sum(axis=1) > 0
).astype(int)

threat_flagged_comments = threat_error_analysis[
    threat_error_analysis["predicted"] == 1
]

threat_moderation_precision = (
    threat_flagged_comments["actual_harmful"].mean()
)

print(
    "Threat Moderation Precision:",
    round(threat_moderation_precision, 3)
)

Threat Moderation Precision: 0.987


In [142]:
hybrid_predictions = pd.DataFrame(
    index=y_validation.index
)

for _, row in hybrid_results_df.iterrows():
    label = row["Label"]
    threshold = row["Threshold"]

    if row["Selected Model"] == "Balanced":
        model = balanced_models[label]
    else:
        model = development_models[label]

    probabilities = model.predict_proba(
        X_validation_tfidf
    )[:, 1]

    hybrid_predictions[label] = (
        probabilities >= threshold
    ).astype(int)

In [143]:
hybrid_predictions.head()

,toxic,severe_toxic,obscene,threat,insult,identity_hate
156856,0,0,0,0,0,0
53862,0,0,0,0,0,0
114241,0,0,0,0,0,0
48359,0,0,0,0,0,0
5675,1,0,1,0,1,0


In [144]:
actual_harmful = (
    y_validation[label_columns].sum(axis=1) > 0
).astype(int)

predicted_harmful = (
    hybrid_predictions[label_columns].sum(axis=1) > 0
).astype(int)

In [145]:
moderation_precision = precision_score(
    actual_harmful,
    predicted_harmful,
    zero_division=0
)

moderation_recall = recall_score(
    actual_harmful,
    predicted_harmful,
    zero_division=0
)

moderation_f1 = f1_score(
    actual_harmful,
    predicted_harmful,
    zero_division=0
)

print("Moderation Precision:", round(moderation_precision, 3))
print("Moderation Recall:", round(moderation_recall, 3))
print("Moderation F1:", round(moderation_f1, 3))

Moderation Precision: 0.819
Moderation Recall: 0.727
Moderation F1: 0.77


In [146]:
moderation_cm = confusion_matrix(
    actual_harmful,
    predicted_harmful
)

moderation_cm

array([[22518,   418],
       [  708,  1888]])

In [147]:
tn, fp, fn, tp = moderation_cm.ravel()

print("True Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)

True Negatives: 22518
False Positives: 418
False Negatives: 708
True Positives: 1888


In [148]:
default_predictions = pd.DataFrame(
    index=y_validation.index
)

for label in label_columns:
    default_predictions[label] = development_models[label].predict(
        X_validation_tfidf
    )

In [149]:
default_predicted_harmful = (
    default_predictions[label_columns].sum(axis=1) > 0
).astype(int)

default_moderation_precision = precision_score(
    actual_harmful,
    default_predicted_harmful,
    zero_division=0
)

default_moderation_recall = recall_score(
    actual_harmful,
    default_predicted_harmful,
    zero_division=0
)

default_moderation_f1 = f1_score(
    actual_harmful,
    default_predicted_harmful,
    zero_division=0
)

print(
    "Default Moderation Precision:",
    round(default_moderation_precision, 3)
)
print(
    "Default Moderation Recall:",
    round(default_moderation_recall, 3)
)
print(
    "Default Moderation F1:",
    round(default_moderation_f1, 3)
)

Default Moderation Precision: 0.947
Default Moderation Recall: 0.57
Default Moderation F1: 0.712


### Moderation-Level Evaluation

In addition to evaluating each harmful-content category separately, the hybrid model was evaluated as an overall moderation system. A comment was considered harmful if at least one harmful-content category was present.

The default model achieved moderation precision of 0.947, recall of 0.570, and F1 of 0.712. This indicates that the default model was highly precise but conservative, missing a substantial portion of harmful comments.

The hybrid model achieved moderation precision of 0.819, recall of 0.727, and F1 of 0.770. Compared with the default system, the hybrid approach substantially increased the proportion of harmful comments detected while maintaining relatively strong precision.

The moderation-level results were stronger than the category-level macro results because some category-level false positives were still harmful comments assigned to a different harmful-content category. For example, 97.14% of the threat classifier's false positives were labeled harmful under at least one other category.

These findings support a tiered moderation strategy. Higher-confidence predictions could support potential HIDE decisions, while the higher-recall hybrid predictions could identify additional comments for human REVIEW. Comments not meeting either condition could be ALLOWed.

In [150]:
moderation_action = pd.Series(
    "ALLOW",
    index=y_validation.index
)

moderation_action[
    predicted_harmful == 1
] = "REVIEW"

moderation_action[
    default_predicted_harmful == 1
] = "HIDE_CANDIDATE"

In [151]:
moderation_action.value_counts()

ALLOW             23226
HIDE_CANDIDATE     1564
REVIEW              742
Name: count, dtype: int64

In [152]:
moderation_action.value_counts(
    normalize=True
).mul(100).round(2)

ALLOW             90.97
HIDE_CANDIDATE     6.13
REVIEW             2.91
Name: proportion, dtype: float64

In [153]:
action_analysis = pd.DataFrame({
    "action": moderation_action,
    "actual_harmful": actual_harmful
})

action_quality = pd.crosstab(
    action_analysis["action"],
    action_analysis["actual_harmful"]
)

action_quality.columns = [
    "Non-Harmful",
    "Harmful"
]

action_quality

,Non-Harmful,Harmful
action,,
ALLOW,22518,708
HIDE_CANDIDATE,83,1481
REVIEW,335,407


In [154]:
action_harmful_rate = (
    action_analysis
    .groupby("action")["actual_harmful"]
    .mean()
    .mul(100)
    .round(2)
)

action_harmful_rate

action
ALLOW              3.05
HIDE_CANDIDATE    94.69
REVIEW            54.85
Name: actual_harmful, dtype: float64

### Moderation Action Analysis

The model outputs were translated into three potential moderation actions: ALLOW, REVIEW, and HIDE_CANDIDATE.

Of the 25,532 validation comments, 90.97% were assigned to ALLOW, 2.91% to REVIEW, and 6.13% to HIDE_CANDIDATE.

The three action groups showed substantially different levels of harmful content. Only 3.05% of comments assigned to ALLOW were labeled harmful, compared with 54.85% of REVIEW comments and 94.69% of HIDE_CANDIDATE comments.

The REVIEW group contained 742 comments, including 407 harmful and 335 non-harmful comments. This mixed composition supports using human review for uncertain predictions.

The HIDE_CANDIDATE group contained 1,481 harmful comments and 83 non-harmful comments. Although this group had a high harmful-content rate of 94.69%, the presence of false positives suggests that automatic hiding should be evaluated carefully before deployment.

The results demonstrate how model predictions can be translated into a tiered moderation policy rather than relying on a single binary harmful/not-harmful decision.

In [155]:
missed_harmful = y_validation[
    (moderation_action == "ALLOW") &
    (actual_harmful == 1)
]

missed_harmful.shape

(708, 6)

In [156]:
missed_by_category = missed_harmful[label_columns].sum()

missed_by_category.sort_values(
    ascending=False
)

toxic            636
obscene          153
insult           137
identity_hate     28
threat            13
severe_toxic       5
dtype: int64

In [157]:
category_miss_analysis = pd.DataFrame({
    "Total Harmful Examples": y_validation[label_columns].sum(),
    "Missed in ALLOW": missed_harmful[label_columns].sum()
})

category_miss_analysis["Miss Rate (%)"] = (
    category_miss_analysis["Missed in ALLOW"] /
    category_miss_analysis["Total Harmful Examples"]
    * 100
)

category_miss_analysis.round(2)

,Total Harmful Examples,Missed in ALLOW,Miss Rate (%)
toxic,2461,636,25.84
severe_toxic,248,5,2.02
obscene,1360,153,11.25
threat,84,13,15.48
insult,1232,137,11.12
identity_hate,241,28,11.62


In [158]:
missed_label_counts = missed_harmful[label_columns].sum(axis=1)

missed_label_counts.value_counts().sort_index()

1    526
2    114
3     55
4     12
5      1
Name: count, dtype: int64

In [159]:
single_label_misses = missed_harmful[
    missed_harmful[label_columns].sum(axis=1) == 1
]

single_label_misses[label_columns].sum().sort_values(
    ascending=False
)

toxic            466
obscene           30
insult            20
identity_hate      7
threat             3
severe_toxic       0
dtype: int64

In [160]:
single_toxic_indices = single_label_misses[
    single_label_misses["toxic"] == 1
].index

X_validation.loc[single_toxic_indices].head(10)

109327    IMHO, the attempt to have this image deleted w...
142045    silencing dissent? \n\nWhat the hell is with y...
79856     "\n\nIT IS TEH SAME DANCE. IT IS JUST BECAUSE ...
50915     PICTURE \n\nThat photo of him makes him look e...
54495     "\n\nWHAT ADVERTISMENT DID I PUT? I can't beli...
29281     Perhaps there's a wikipedia for adults somewhe...
147502    Respect is earned by respect \n\nThat user IS ...
62413     impact page \n\nyou might as well just give up...
63517     LISTEN UP WIKI-KIKES, \n\nBurn this IP, it mea...
30851     Hi babe I just wanted to let you know that I m...
Name: comment_text, dtype: str

In [161]:
toxic_probabilities = development_models["toxic"].predict_proba(
    X_validation_tfidf
)[:, 1]

toxic_probability_series = pd.Series(
    toxic_probabilities,
    index=y_validation.index
)

missed_single_toxic_probabilities = (
    toxic_probability_series.loc[single_toxic_indices]
)

missed_single_toxic_probabilities.describe()

count    466.000000
mean       0.130614
std        0.068916
min        0.003294
25%        0.076548
50%        0.121028
75%        0.181205
max        0.268185
dtype: float64

In [162]:
for threshold in [0.25, 0.20, 0.15, 0.10, 0.05]:
    caught = (
        missed_single_toxic_probabilities >= threshold
    ).sum()

    print(
        f"Threshold {threshold}: "
        f"{caught} of {len(missed_single_toxic_probabilities)} "
        f"would be caught"
    )

Threshold 0.25: 26 of 466 would be caught
Threshold 0.2: 90 of 466 would be caught
Threshold 0.15: 168 of 466 would be caught
Threshold 0.1: 290 of 466 would be caught
Threshold 0.05: 405 of 466 would be caught


### Error Analysis

Error analysis was conducted on the 708 harmful validation comments assigned to ALLOW by the hybrid moderation system.

The largest source of missed harmful content was the toxic category. Of 2,461 toxic validation examples, 636 were missed, corresponding to a 25.84% miss rate. In comparison, the miss rates for obscene, insult, identity hate, threat, and severe toxic were substantially lower.

Most missed harmful comments contained only one harmful-content label. Of the 708 missed comments, 526 had only one positive label, and 466 of these were labeled only as toxic. This suggests that multi-label harmful comments are easier for the moderation system to detect because multiple category classifiers provide opportunities to flag them.

The predicted toxic probabilities for the 466 missed toxic-only comments were also examined. Their median predicted probability was 0.121 and their mean was 0.131, compared with the selected toxic classification threshold of 0.27. Only 26 of the 466 comments would be recovered by slightly lowering the threshold to 0.25. Recovering substantially more comments would require much lower thresholds, which would likely increase false positives.

These findings suggest that threshold adjustment alone may not address the remaining toxic-content detection errors. The results provide motivation for evaluating a contextual language model and comparing its performance with the TF-IDF and Logistic Regression baseline.